In [1]:
# ======================
# cargar archivo completo
# ======================

import pandas as pd
import numpy as np

file_path = 'Global Superstore.xlsx'
xl = pd.ExcelFile(file_path)

In [2]:
# ========================================
# Cargar cada hoja en un DataFrame inicial
# ========================================

df_orders = pd.read_excel(file_path, sheet_name = 'Orders')
df_returns = pd.read_excel(file_path, sheet_name = 'Returns')
df_people = pd.read_excel(file_path, sheet_name = 'People')

In [3]:
# ========================================
# validación de seguridad o "sanity check" 
# ========================================
# assert (verifica si una condición es verdadera)

assert not df_orders.empty, "La tabla Orders está vacía"
assert not df_returns.empty, "La tabla Returns está vacía"
assert not df_people.empty, "La tabla People está vacía"

In [4]:
# ==========================================================================
# Sustituir nulos en Postal Code y tratarlo como variable categórica textual
# ==========================================================================

df_orders['Postal Code'] = (
    df_orders['Postal Code']
    .fillna('desconocido')
    .astype(str)
    .str.strip()
)

In [6]:
# =========================
# 1. NORMALIZACIÓN DE FECHAS
# =========================
# Conversión explícita a datetime (no asumir tipos)
df_orders['Order Date'] = pd.to_datetime(df_orders['Order Date'], errors='coerce')
df_orders['Ship Date']  = pd.to_datetime(df_orders['Ship Date'], errors='coerce')

# Validación básica de fechas
# (Ship Date nunca debería ser anterior a Order Date)
df_orders = df_orders[df_orders['Ship Date'] >= df_orders['Order Date']]

In [7]:
# =======================================
# 2. NORMALIZACIÓN DE VARIABLES NUMÉRICAS
# =======================================

cols_numericas = ['Sales', 'Profit', 'Shipping Cost']

for col in cols_numericas:
    df_orders[col] = (
        df_orders[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

In [8]:
# =======================================
# 3. TRATAMIENTO DE VARIABLES CATEGÓRICAS
# =======================================

cols_categoricas = [
    'Ship Mode', 'Segment', 'Market',
    'Region', 'Category', 'Order Priority'
]

for col in cols_categoricas:
    df_orders[col] = (
        df_orders[col]
        .astype(str)
        .str.strip()
        .astype('category')
    )

In [9]:
# =====================================
# 4. CLAVES Y VARIABLES IDENTIFICADORAS
# =====================================

# Customer ID y Order ID como string (nunca numéricas)
df_orders['Customer ID'] = df_orders['Customer ID'].astype(str).str.strip()
df_orders['Order ID']    = df_orders['Order ID'].astype(str).str.strip()

# Postal Code no es numérico a efectos analíticos
df_orders['Postal Code'] = (
    df_orders['Postal Code']
    .fillna('desconocido')
    .astype(str)
    .str.strip()
)

In [10]:
# ===========================================================================
# 5. Limpiar espacios en blanco en los nombres de las columnas (por si acaso)
# ===========================================================================
df_orders.columns = df_orders.columns.str.strip()
df_returns.columns = df_returns.columns.str.strip()

In [11]:
# =========================
# 6. CHEQUEO FINAL DE TIPOS
# =========================
df_orders.dtypes

Row ID                     int64
Order ID                  object
Order Date        datetime64[ns]
Ship Date         datetime64[ns]
Ship Mode               category
Customer ID               object
Customer Name             object
Segment                 category
City                      object
State                     object
Country                   object
Postal Code               object
Market                  category
Region                  category
Product ID                object
Category                category
Sub-Category              object
Product Name              object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Shipping Cost            float64
Order Priority          category
dtype: object

In [12]:
# ============================================================
# PASO 1: FILTRADO B2B – SOLO SEGMENTO CORPORATE
# ============================================================

df_orders = df_orders[df_orders['Segment'] == 'Corporate'].copy()

print(f"Registros Corporate: {df_orders.shape[0]}")
print(f"Clientes únicos Corporate: {df_orders['Customer ID'].nunique()}")

Registros Corporate: 15429
Clientes únicos Corporate: 476


In [13]:
# ============================================================
# PASO 2: VARIABLES BASE A NIVEL PEDIDO
# ============================================================

# Conversión de fechas
df_orders['Order Date'] = pd.to_datetime(df_orders['Order Date'])
df_orders['Ship Date'] = pd.to_datetime(df_orders['Ship Date'])

# Latencia logística por pedido
df_orders['delivery_days'] = (
    df_orders['Ship Date'] - df_orders['Order Date']
).dt.days

# Flag de envío urgente
df_orders['is_urgent'] = df_orders['Ship Mode'].isin([
    'Same Day', 'First Class'
]).astype(int)

# Flag de prioridad alta / crítica
df_orders['is_high_priority'] = df_orders['Order Priority'].isin([
    'High', 'Critical'
]).astype(int)


In [14]:
# ============================================================
# PASO 3: INCORPORACIÓN DE DEVOLUCIONES
# ============================================================

# Asumimos df_returns con columna 'Order ID'
# devoluciones a nivel pedido

df_returns['is_returned'] = 1

df_orders = df_orders.merge(
    df_returns[['Order ID', 'is_returned']],
    on='Order ID',
    how='left'
)

df_orders['is_returned'] = df_orders['is_returned'].fillna(0)


In [15]:
# ============================================================
# PASO 4: CONSTRUCCIÓN DE LA TABLA CUSTOMER 360
# 1 fila = 1 cliente
# ============================================================
# aquí se cumple: cambio de granularidad, métricas agregadas correctamente, base única para todo el proyecto

customer_360 = (
    df_orders
    .groupby('Customer ID')
    .agg(
        # ======================
        # EJE VALOR
        # ======================
        total_sales=('Sales', 'sum'),
        total_profit=('Profit', 'sum'),
        num_orders=('Order ID', 'nunique'),

        # ======================
        # EJE RIESGO
        # ======================
        last_order_date=('Order Date', 'max'),
        avg_delivery_days=('delivery_days', 'mean'),
        return_rate=('is_returned', 'mean'),

        # ======================
        # EJE COSTE / ESFUERZO
        # ======================
        total_shipping_cost=('Shipping Cost', 'sum'),
        urgent_ship_ratio=('is_urgent', 'mean'),
        high_priority_ratio=('is_high_priority', 'mean')
    )
    .reset_index()
)

In [19]:
# ============================================================
# PASO 5: MÉTRICAS DERIVADAS (POR CLIENTE)
# ============================================================

# Ticket medio
customer_360['avg_ticket'] = (
    customer_360['total_sales'] / customer_360['num_orders']
)

# Margen %
customer_360['margin_pct'] = (
    customer_360['total_profit'] / customer_360['total_sales']
)

# Recencia (días)
dataset_end_date = df_orders['Order Date'].max()

customer_360['recency_days'] = (
    dataset_end_date - customer_360['last_order_date']
).dt.days

# Ratio coste logístico
customer_360['shipping_cost_ratio'] = (
    customer_360['total_shipping_cost'] / customer_360['total_sales']
)

In [20]:
# ============================================================
# PASO 6: VALIDACIÓN FINAL DE LA TABLA CUSTOMER 360
# ============================================================

print(f"Filas (clientes): {customer_360.shape[0]}")
print(f"Customer ID únicos: {customer_360['Customer ID'].nunique()}")

customer_360.head()


Filas (clientes): 476
Customer ID únicos: 476


,Customer ID,total_sales,total_profit,num_orders,last_order_date,avg_delivery_days,return_rate,total_shipping_cost,urgent_ship_ratio,high_priority_ratio,avg_ticket,margin_pct,recency_days,shipping_cost_ratio
0,AB-10600,13318.66310,905.36070,22,2014-11-11,3.710526,0.026316,1375.881,0.210526,0.394737,605.393777,0.067977,50,0.103305
1,AB-600,5408.55000,964.65000,5,2014-12-24,3.038462,0.000000,1311.720,0.692308,0.153846,1081.710000,0.178356,7,0.242527
2,AC-10420,10607.69470,206.48050,24,2014-12-25,4.400000,0.025000,749.061,0.150000,0.275000,441.987279,0.019465,6,0.070615
3,AC-10615,12729.72448,1160.67578,29,2014-12-25,4.553846,0.092308,1271.200,0.015385,0.369231,438.956017,0.091178,6,0.099861
4,AC-420,1689.67500,237.55500,9,2014-12-18,3.952381,0.000000,120.820,0.095238,0.380952,187.741667,0.140592,13,0.071505


In [22]:
# ============================================================
# EJE VALOR – MÉTRICAS ECONÓMICAS POR CLIENTE
# ============================================================

value_metrics = (
    df_orders
    .groupby('Customer ID')
    .agg(
        total_sales=('Sales', 'sum'),
        total_profit=('Profit', 'sum'),
        num_orders=('Order ID', 'nunique')
    )
)

# Métricas derivadas de valor
value_metrics['margin_pct'] = (
    value_metrics['total_profit'] / value_metrics['total_sales']
)

value_metrics['avg_ticket'] = (
    value_metrics['total_sales'] / value_metrics['num_orders']
)

value_metrics.head()


,total_sales,total_profit,num_orders,margin_pct,avg_ticket
Customer ID,,,,,
AB-10600,13318.66310,905.36070,22,0.067977,605.393777
AB-600,5408.55000,964.65000,5,0.178356,1081.710000
AC-10420,10607.69470,206.48050,24,0.019465,441.987279
AC-10615,12729.72448,1160.67578,29,0.091178,438.956017
AC-420,1689.67500,237.55500,9,0.140592,187.741667


In [24]:
# ============================================================
# EJE RIESGO – ESTABILIDAD Y DETERIORO
# ============================================================

risk_metrics = (
    df_orders
    .groupby('Customer ID')
    .agg(
        last_order_date=('Order Date', 'max'),
        avg_delivery_days=('delivery_days', 'mean'),
        return_rate=('is_returned', 'mean')
    )
)

# Recencia (días desde la última compra)
dataset_end_date = df_orders['Order Date'].max()

risk_metrics['recency_days'] = (
    dataset_end_date - risk_metrics['last_order_date']
).dt.days

risk_metrics.head()


,last_order_date,avg_delivery_days,return_rate,recency_days
Customer ID,,,,
AB-10600,2014-11-11,3.710526,0.026316,50
AB-600,2014-12-24,3.038462,0.000000,7
AC-10420,2014-12-25,4.400000,0.025000,6
AC-10615,2014-12-25,4.553846,0.092308,6
AC-420,2014-12-18,3.952381,0.000000,13


In [25]:
# ============================================================
# EJE ESFUERZO / COSTE DE SERVICIO
# ============================================================

effort_metrics = (
    df_orders
    .groupby('Customer ID')
    .agg(
        total_shipping_cost=('Shipping Cost', 'sum'),
        urgent_ship_ratio=('is_urgent', 'mean'),
        high_priority_ratio=('is_high_priority', 'mean')
    )
)

effort_metrics.head()


,total_shipping_cost,urgent_ship_ratio,high_priority_ratio
Customer ID,,,
AB-10600,1375.881,0.210526,0.394737
AB-600,1311.720,0.692308,0.153846
AC-10420,749.061,0.150000,0.275000
AC-10615,1271.200,0.015385,0.369231
AC-420,120.820,0.095238,0.380952


In [26]:
# ============================================================
# CUSTOMER 360 FINAL – UNIÓN DE LOS 3 EJES
# ============================================================

customer_360 = (
    value_metrics
    .join(risk_metrics, how='inner')
    .join(effort_metrics, how='inner')
    .reset_index()
)

customer_360.head()


,Customer ID,total_sales,total_profit,num_orders,margin_pct,avg_ticket,last_order_date,avg_delivery_days,return_rate,recency_days,total_shipping_cost,urgent_ship_ratio,high_priority_ratio
0,AB-10600,13318.66310,905.36070,22,0.067977,605.393777,2014-11-11,3.710526,0.026316,50,1375.881,0.210526,0.394737
1,AB-600,5408.55000,964.65000,5,0.178356,1081.710000,2014-12-24,3.038462,0.000000,7,1311.720,0.692308,0.153846
2,AC-10420,10607.69470,206.48050,24,0.019465,441.987279,2014-12-25,4.400000,0.025000,6,749.061,0.150000,0.275000
3,AC-10615,12729.72448,1160.67578,29,0.091178,438.956017,2014-12-25,4.553846,0.092308,6,1271.200,0.015385,0.369231
4,AC-420,1689.67500,237.55500,9,0.140592,187.741667,2014-12-18,3.952381,0.000000,13,120.820,0.095238,0.380952


In [27]:
# ============================================================
# VALIDACIÓN FINAL CUSTOMER 360
# ============================================================

print(f"Nº filas (clientes): {customer_360.shape[0]}")
print(f"Customer ID únicos: {customer_360['Customer ID'].nunique()}")

customer_360.info()


Nº filas (clientes): 476
Customer ID únicos: 476
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 476 entries, 0 to 475
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Customer ID          476 non-null    object        
 1   total_sales          476 non-null    float64       
 2   total_profit         476 non-null    float64       
 3   num_orders           476 non-null    int64         
 4   margin_pct           476 non-null    float64       
 5   avg_ticket           476 non-null    float64       
 6   last_order_date      476 non-null    datetime64[ns]
 7   avg_delivery_days    476 non-null    float64       
 8   return_rate          476 non-null    float64       
 9   recency_days         476 non-null    int64         
 10  total_shipping_cost  476 non-null    float64       
 11  urgent_ship_ratio    476 non-null    float64       
 12  high_priority_ratio  476 non-null    float6

In [28]:
# ============================================================
# VALIDACIÓN: PEDIDOS VS LÍNEAS DE PEDIDO
# ============================================================

total_rows = df_orders.shape[0]
unique_orders = df_orders['Order ID'].nunique()

print(f"Líneas de pedido: {total_rows}")
print(f"Pedidos únicos: {unique_orders}")


Líneas de pedido: 15429
Pedidos únicos: 7673


In [29]:
# ============================================================
# VALIDACIÓN: RECUENTO CORRECTO DE PEDIDOS POR CLIENTE
# ============================================================

check_orders = (
    df_orders
    .groupby('Customer ID')
    .agg(
        lines=('Order ID', 'count'),
        distinct_orders=('Order ID', 'nunique')
    )
    .query('lines != distinct_orders')
    .head()
)

check_orders


,lines,distinct_orders
Customer ID,,
AB-10600,38,22
AB-600,26,5
AC-10420,40,24
AC-10615,65,29
AC-420,21,9


In [30]:
# ============================================================
# VALIDACIÓN: ÚLTIMA FECHA DE PEDIDO Y RECENCIA
# ============================================================

#Permite detectar: clientes inactivo, extremos lógicos, coherencia temporal

customer_360[['Customer ID', 'last_order_date', 'recency_days']].sort_values(
    'recency_days', ascending=False
).head()

,Customer ID,last_order_date,recency_days
152,DW-3195,2012-02-02,1063
254,JS-5595,2013-02-20,679
316,MC-7635,2013-03-07,664
113,CY-2745,2013-08-26,492
172,EM-3825,2013-09-05,482


In [31]:
# ============================================================
# VALIDACIÓN: LATENCIAS LOGÍSTICAS
# ============================================================

# buscamos: valores negativos y valores absurdos (>60 días)

customer_360['avg_delivery_days'].describe()

count    476.000000
mean       3.967901
std        0.703297
min        0.727273
25%        3.665000
50%        4.000000
75%        4.333333
max        7.000000
Name: avg_delivery_days, dtype: float64

In [32]:
# ============================================================
# CÁLCULO DE RATIOS DERIVADOS (OBLIGATORIOS)
# ============================================================

# Ratio coste logístico / ventas
customer_360['shipping_cost_ratio'] = (
    customer_360['total_shipping_cost'] / customer_360['total_sales']
)

In [33]:
# ============================================================
# VALIDACIÓN: RATIOS Y PORCENTAJES
# ============================================================

ratio_cols = [
    'margin_pct',
    'return_rate',
    'urgent_ship_ratio',
    'high_priority_ratio',
    'shipping_cost_ratio'
]

customer_360[ratio_cols].describe()


,margin_pct,return_rate,urgent_ship_ratio,high_priority_ratio,shipping_cost_ratio
count,476.000000,476.000000,476.000000,476.000000,476.000000
mean,0.074419,0.038184,0.195705,0.369727,0.109656
std,0.229198,0.061295,0.157987,0.181588,0.031640
min,-1.595228,0.000000,0.000000,0.000000,0.039569
25%,0.049942,0.000000,0.090909,0.250000,0.088048
50%,0.126257,0.000000,0.174802,0.357143,0.103959
75%,0.181868,0.065847,0.267161,0.461538,0.124652
max,0.451063,0.380000,1.000000,1.000000,0.254072


In [34]:
# ============================================================
# CONTROL DE VALORES INFINITOS O NULOS EN RATIOS
# ============================================================

customer_360[ratio_cols].isna().sum(), 
customer_360[ratio_cols].isin([float('inf'), float('-inf')]).sum()


margin_pct             0
return_rate            0
urgent_ship_ratio      0
high_priority_ratio    0
shipping_cost_ratio    0
dtype: int64

In [36]:
output_file = "Dataset_normalizado_resultado.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_orders.to_excel(writer, sheet_name="Orders", index=False)
    df_returns.to_excel(writer, sheet_name="Returns", index=False)
    df_people.to_excel(writer, sheet_name="People", index=False)